In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [12]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_spring_2026_svg_generation_from_text_prompts_extended_deadline_path = kagglehub.competition_download('dl-spring-2026-svg-generation-from-text-prompts-extended-deadline')

print('Data source import complete.')


Data source import complete.


### References
Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune

In [13]:
# Install unsloth
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
# Pin trl to 0.18.2 — fixes "int has no attribute mean" bug with unsloth
%pip install -q "trl==0.18.2"

In [14]:
# Imports and Seedings
import os, re, time, random, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
Tesla T4


In [15]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
#CHANGE FOR FINAL TESTING
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "model_name":                   "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "max_seq_length":               1024,  # second iteration change from 1024 to 2048 to capture longer svgs
    "lora_r":                       16,    # was 8
    "lora_alpha":                   32,    # was 16
    "learning_rate":                2e-4,
    "num_train_epochs":             1,     # second iteration change from 1 to 2 to train longer
    "per_device_train_batch_size":  1,
    "gradient_accumulation_steps":  8,
    "warmup_steps":                 50,
    "weight_decay":                 0.01,
    "logging_steps":                20,
    "save_steps":                   200,
    "n_train_samples":              5000, # second iteration change from 3k to 6k to get better results
    "eval_size":                    0.02,
    #"output_dir": "/kaggle/working/qwen_svg_lora"
    "output_dir": "/content/drive/MyDrive/qwen_svg_lora",  # save to Drive
}
CONFIG

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 1024,
 'lora_r': 16,
 'lora_alpha': 32,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'warmup_steps': 50,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'save_steps': 200,
 'n_train_samples': 5000,
 'eval_size': 0.02,
 'output_dir': '/content/drive/MyDrive/qwen_svg_lora'}

In [16]:
import os
print(os.listdir(dl_spring_2026_svg_generation_from_text_prompts_extended_deadline_path))

['sample_submission.csv', 'test.csv', 'train.csv']


In [17]:
# Load data CSVs

#DATA_DIR = "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline"
DATA_DIR = dl_spring_2026_svg_generation_from_text_prompts_extended_deadline_path

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

# Small subset for this first run
train_df = train_df.sample(n=CONFIG["n_train_samples"], random_state=SEED).reset_index(drop=True)

print(f"Train rows: {len(train_df)}  |  Test rows: {len(test_df)}")
print(train_df.head(2))

Train rows: 5000  |  Test rows: 1000
                                 id  \
0  bdcb4c0f87a075134b991bb5c9ac0dec   
1  73d39ee0485b86ecc66ccc4875a0c2d6   

                                              prompt  \
0  A black icon featuring a vertical rectangle wi...   
1  A single black outline of a heart shape agains...   

                                                 svg  
0  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
1  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  


In [18]:
# Basic SVG Validation

ALLOWED_TAGS = {
    "svg","g","path","rect","circle","ellipse","line","polyline","polygon",
    "defs","use","symbol","clipPath","mask","linearGradient","radialGradient",
    "stop","text","tspan","title","desc","style","pattern","marker","filter"
}

def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or not svg_text.strip().lower().startswith("<svg"):
        return False
    if len(svg_text) > 16000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    path_count = sum(1 for el in root.iter() if el.tag.split("}")[-1] == "path")
    if path_count > 256:
        return False
    return True

# Drop training rows with invalid SVGs so we don't train on garbage
before = len(train_df)
train_df = train_df[train_df["svg"].apply(is_valid_svg)].reset_index(drop=True)
print(f"Valid SVG rows: {len(train_df)} / {before}")

Valid SVG rows: 5000 / 5000


In [19]:
# Format as Chat Text
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element, "
    "max 256×256 canvas, max 256 path elements."
)

def make_chat_text(row):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{row['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{row['svg']}<|im_end|>"
    )

train_df["text"] = train_df.apply(make_chat_text, axis=1)

# Train / eval split
eval_n   = max(1, int(len(train_df) * CONFIG["eval_size"]))
eval_df  = train_df.iloc[:eval_n].reset_index(drop=True)
train_df = train_df.iloc[eval_n:].reset_index(drop=True)

train_hf = Dataset.from_dict({"text": train_df["text"].tolist()})
eval_hf  = Dataset.from_dict({"text": eval_df["text"].tolist()})

print(f"Training samples: {len(train_hf)}  |  Eval samples: {len(eval_hf)}")
print("\nSample (first 300 chars):")
print(train_hf[0]["text"][:300])

Training samples: 4900  |  Eval samples: 100

Sample (first 300 chars):
<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element, max 256×256 canvas, max 256 path elements.<|im_end|>
<|im_start|>user
The image features a black outline of an eraser with a small, darker black smudge or mark to its 


In [20]:
# Load Model + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = CONFIG["model_name"],
    max_seq_length  = CONFIG["max_seq_length"],
    dtype           = None,
    load_in_4bit    = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                        = CONFIG["lora_r"],
    lora_alpha               = CONFIG["lora_alpha"],
    lora_dropout             = 0,
    bias                     = "none",
    target_modules           = ["q_proj","k_proj","v_proj","o_proj",
                                "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state             = SEED,
)

==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [21]:
# Train
from trl import SFTTrainer, SFTConfig
# Clear stale unsloth compiled cache
import shutil
shutil.rmtree("/kaggle/working/unsloth_compiled_cache", ignore_errors=True)
print("Cache cleared")


training_args = SFTConfig(
    output_dir                  = CONFIG["output_dir"],
    num_train_epochs            = CONFIG["num_train_epochs"],
    per_device_train_batch_size = CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"],
    learning_rate               = CONFIG["learning_rate"],
    warmup_steps                = 50,
    weight_decay                = CONFIG["weight_decay"],
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = CONFIG["logging_steps"],
    eval_strategy               = "no",   # disabled — eval crashes on long sequences
    #eval_strategy               = "steps",
    #eval_steps                  = CONFIG["eval_steps"],
    save_steps                  = CONFIG["save_steps"],
    save_total_limit            = 1,
    report_to                   = "none",
    optim                       = "paged_adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = SEED,
    max_seq_length              = CONFIG["max_seq_length"],
    dataset_text_field          = "text",
    packing                     = False, # set to true to make faster? but gives error
)


trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_hf,
    eval_dataset  = eval_hf,
    args          = training_args,
)

print("Starting training …")
train_result = trainer.train()
print(train_result)

Cache cleared


/content/unsloth_compiled_cache/UnslothSFTTrainer.py:786: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4900 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training …


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,900 | Num Epochs = 1 | Total steps = 613
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
20,0.990198
40,0.656125
60,0.618245
80,0.545189
100,0.522189
120,0.488361
140,0.493171
160,0.484665
180,0.442741
200,0.430923


TrainOutput(global_step=613, training_loss=0.4792259412333005, metrics={'train_runtime': 2298.6059, 'train_samples_per_second': 2.132, 'train_steps_per_second': 0.267, 'total_flos': 3.5607657486495744e+16, 'train_loss': 0.4792259412333005})


In [22]:
# Save model
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Adapter saved to {CONFIG['output_dir']}")
print(os.listdir(CONFIG["output_dir"]))

Adapter saved to /content/drive/MyDrive/qwen_svg_lora
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'checkpoint-613']


In [23]:
model.save_pretrained_merged(
    "/content/drive/MyDrive/svg_competition_v2/qwen_svg_merged_v2",
    tokenizer,
    save_method = "merged_16bit",
)
print("Merged model saved ")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:28<00:00, 28.11s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:20<00:00, 80.83s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/svg_competition_v2/qwen_svg_merged_v2`
Merged model saved 
